# Projections climatiques Socle Métropole (Météo-France 2025) — 5 communes pilotes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarcoDamz/bati-scolaire-climat/blob/claude/enrich-demo-instances-oncka4/notebooks/01_extraction_climat.ipynb)

**Source** : [Projections climatiques Socle Métropole](https://www.data.gouv.fr/datasets/projections-climatiques-pour-le-hackathon-2025-socle-metropole)
(Météo-France, Licence Ouverte, sans compte), stockage objet data.gouv.

- **RCM** : 17 simulations EURO-CORDEX CMIP6, quotidiennes, 8 km, corrigées (CDF-t) sur ANASTASIA/SAFRAN,
  historique + SSP3-7.0 (+ 4 SSP5-8.5) ; températures, pluie, vent, humidité, rayonnement.
- **CPRCM** (option) : AROME 2,5 km à convection résolue, **pluie horaire** (orages méditerranéens).

Seules les mailles qui contiennent les 228 bâtiments des pilotes sont lues (lecture partielle à distance).
Sortie : `Mon Drive/SaaS adaptation climatique/demo/donnees/climat_socle/`. Reprise automatique après coupure.

## 1. Paramètres et Google Drive

In [ ]:
DOSSIER_DRIVE = "/content/drive/MyDrive/SaaS adaptation climatique/demo/donnees/climat_socle"
AVEC_PLUIE_HORAIRE_AROME = True   # CPRCM 2,5 km : plus long, mais indispensable pour les orages (Aude, Var)
DEPOT = "https://github.com/MarcoDamz/bati-scolaire-climat.git"
BRANCHE = "claude/enrich-demo-instances-oncka4"   # passer à "main" une fois la branche fusionnée

from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DOSSIER_DRIVE, exist_ok=True)

## 2. Installation et code depuis GitHub

In [ ]:
!pip install -q xarray h5netcdf h5py netCDF4 fsspec aiohttp pyproj
import os, sys, subprocess, importlib
CODE = "/content/bati-scolaire-climat"
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin", BRANCHE], check=True)
    subprocess.run(["git", "-C", CODE, "checkout", "-q", "-B", BRANCHE, f"origin/{BRANCHE}"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCHE, DEPOT, CODE], check=True)
sys.path.insert(0, f"{CODE}/phase1")
import telecharger_socle as t
importlib.reload(t)

## 3. Accès et inventaire (quelques minutes)

In [ ]:
assert t.tester_acces(), "Stockage data.gouv inaccessible depuis Colab : lancer phase1/telecharger_socle.py depuis un ordinateur."
selection = t.executer(DOSSIER_DRIVE, lister=True, cprcm=AVEC_PLUIE_HORAIRE_AROME)
selection.groupby(["famille", "exp", "var"])["chaine"].nunique().unstack("var")

**Attendu** : 17 simulations historiques et 17 SSP3-7.0 pour chaque variable RCM (et 4 SSP5-8.5).
Si le tableau est vide ou incomplet, envoyez `socle_inventaire.csv` (dans le dossier Drive) : j'adapterai la
reconnaissance des noms de fichiers.

## 4. Extraction (plusieurs heures, reprise automatique)

In [ ]:
donnees = t.executer(DOSSIER_DRIVE, cprcm=AVEC_PLUIE_HORAIRE_AROME)
donnees.groupby(["famille", "scenario", "variable"], observed=True)["chaine"].nunique().unstack("variable")